# Understanding Learning Curves

In this notebook, we will explore the concept of learning curves and how different hyperparameters, specifically the learning rate, affect the model's training process. We will look at examples of **Overfitting**, **Underfitting**, and a **"Just Right"** learning curve.

First, let's install the necessary libraries for PyTorch and Hugging Face.

In [1]:
# Install Pytorch & other libraries
%pip install -qqq torch torchvision setuptools scikit-learn

# Install Hugging Face libraries
%pip install  --upgrade datasets -qqq accelerate hf-transfer transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 91.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00:00:0100:01


## 1. Loading the Dataset

We'll use the `burtenshaw/PleIAs_common_corpus_code_classification` dataset. This dataset is likely used for classifying code snippets into different domains or languages.

In [2]:
from datasets import load_dataset

# Dataset id from huggingface.co/dataset
dataset_id = "burtenshaw/PleIAs_common_corpus_code_classification"

# Load raw dataset
dataset = load_dataset(dataset_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/418 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/94.9M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/94.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/20.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127723 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/14192 [00:00<?, ? examples/s]

Let's take a quick look at the size of our training set and the first example to understand its structure.

In [3]:
print(len(dataset["train"]))
print(dataset["train"][0])

127723
{'text': '/*\n * Copyright (c) 2000 Kungliga Tekniska Högskolan\n * (Royal Institute of Technology, Stockholm, Sweden).\n * All rights reserved.\n *\n * Redistribution and use in source and binary forms, with or without\n * modification, are permitted provided that the following conditions\n * are met:\n *\n * 1. Redistributions of source code must retain the above copyright\n *    notice, this list of conditions and the following disclaimer.\n *\n * 2. Redistributions in binary form must reproduce the above copyright\n *    notice, this list of conditions and the following disclaimer in the\n *    documentation and/or other materials provided with the distribution.\n *\n * 3. Neither the name of the Institute nor the names of its contributors\n *    may be used to endorse or promote products derived from this software\n *    without specific prior written permission.\n *\n * THIS SOFTWARE IS PROVIDED BY THE INSTITUTE AND CONTRIBUTORS ``AS IS\'\' AND\n * ANY EXPRESS OR IMPLIED W

## 2. Tokenization

To feed text into our model, we must first convert it into numbers using a Tokenizer. We load the tokenizer for `answerdotai/ModernBERT-base` and apply it to our entire dataset. We use dynamic padding and truncation so all sequences in a batch have the same length.

In [4]:
from transformers import AutoTokenizer

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Tokenize helper function
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

tokenized_dataset["train"].features.keys()
# dict_keys(['labels', 'input_ids', 'attention_mask'])


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/127723 [00:00<?, ? examples/s]

Map:   0%|          | 0/14192 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'attention_mask'])

## 3. Label Preparation

Machine learning models require labels to be numerical. Here, we extract all unique labels from our dataset and create mappings from the string labels to integers (`label2id`) and vice versa (`id2label`). This is essential for setting up the classification head on top of the transformer.

In [5]:
from transformers import AutoModelForSequenceClassification

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Prepare model labels - useful for inference
labels = list(set(tokenized_dataset["train"]["labels"]))
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

## 4. Initializing the Model

We load the pretrained `ModernBERT-base` model and add a sequence classification head on top. We pass our `num_labels`, `label2id`, and `id2label` to ensure the model's output configuration matches our specific classification task.

In [6]:
# Download the model from huggingface.co/models
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Defining Evaluation Metrics

We need a way to measure our model's performance during training. We define a `compute_metrics` function that calculates the **F1 score**, a common metric for classification tasks that balances precision and recall.

In [7]:
import numpy as np
from sklearn.metrics import f1_score

# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}


## 6. Setting up Training Arguments

We define the hyperparameter configuration for our training process using `TrainingArguments`. This sets the batch sizes, number of epochs, evaluation strategy, logging frequency, and specifies that we want to load the best model at the end based on the F1 score.

In [11]:
training_args = TrainingArguments(
    output_dir="ModernBERT-code-classifier",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=5e-5,
    num_train_epochs=5,

    bf16=True,
    optim="adamw_torch_fused",

    logging_strategy="steps",
    logging_steps=100,

    eval_strategy="epoch",   # keep OLD name if your version requires it
    save_strategy="epoch",

    save_total_limit=2,
    load_best_model_at_end=True,

    metric_for_best_model="f1",

    push_to_hub=False,
    report_to="none"
)

## 7. Experiment 1: Overfitting

**Overfitting** occurs when a model learns the training data *too well*, including its noise and outliers, but fails to generalize to new, unseen data (the validation/test set). 

In this first experiment, we will deliberately induce overfitting by training on an extremely small subset of the data (only 100 examples) for several epochs. Notice how the training loss decreases rapidly, but the evaluation metrics on the test set will likely plateau or even worsen, indicating the model is memorizing the small dataset instead of learning generalizable features.

In [15]:
limited_dataset = tokenized_dataset["train"].select(range(100))

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

NameError: name 'model' is not defined

Before we move to the next experiment, let's clear the GPU memory and delete the trainer and model instances. This prevents Out-Of-Memory (OOM) errors.

In [14]:
# clear memory

import torch
torch.cuda.empty_cache()

del trainer
del model
del limited_dataset

## 8. Experiment 2: Underfitting

**Underfitting** happens when a model is too simple or hasn't trained enough to learn the underlying patterns in the data. The model performs poorly on both the training data and the test data.

Here, we induce underfitting by setting the learning rate exceptionally low (`1e-7`). The model updates its weights so slowly that it barely learns anything meaningful within the given epochs. The learning curve will show minimal improvement.

In [16]:
# define a low learning rate
training_args.learning_rate = 1e-7

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()


NameError: name 'model' is not defined

Again, we clear the memory to ensure a clean slate for our final experiment.

In [ ]:
# clear memory

import torch
torch.cuda.empty_cache()

del trainer
del model

## 9. Experiment 3: "Just Right"

A **"Just Right"** learning curve shows a steady decrease in training loss and a corresponding improvement in validation metrics. The model learns the general patterns without memorizing the noise.

We achieve this by setting a balanced, optimal learning rate (`5e-5`), which allows the model to take appropriately sized steps toward the optimal weights.

In [ ]:
# define a valid learning rate
training_args.learning_rate = 5e-5

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

## 10. Inference

Now that we understand how training works under different conditions, let's see how a fully trained model is used in practice. We use the Hugging Face `pipeline` to load a completely fine-tuned version of ModernBERT (`argilla/ModernBERT-domain-classifier`) and use it to classify a sample piece of Python code.

In [ ]:
from transformers import pipeline

# load model from huggingface.co/models using our repository id
classifier = pipeline(
    task="text-classification",
    model="argilla/ModernBERT-domain-classifier",
    device=0,
)

sample = """def add_numbers(a, b):
    return a + b"""

classifier(sample)
